# Endothelial receptor mining from GSE225948

This notebook inspects the public single-cell transcriptomic dataset associated with the StrokeVis resource and extracts gene-level endothelial receptor candidates for downstream peptide-shuttle design.

The aim is not to perform full formal differential-expression modelling. Instead, this notebook builds a practical omics-to-peptide-target workflow:

1. inspect the downloaded raw metadata and count files,
2. identify brain endothelial cell populations and experimental groups,
3. verify endothelial/BBB identity using canonical marker genes,
4. build EC pseudobulk expression profiles,
5. rank early stroke-responsive endothelial genes,
6. annotate candidate proteins with UniProt accessions and subcellular localisation,
7. classify putative surface accessibility,
8. export a receptor shortlist for downstream peptide generation and structure-aware triage.

Dataset/resource:

- **Title:** Brain and blood single-cell transcriptomic analysis in acute and subacute phases after experimental stroke  
- **Web resource:** https://anratherlab.shinyapps.io/strokevis/  
- **Public release:** June 14, 2023  
- **Local raw-data folder:** `../data/raw/GSE225948_RAW/`

Important scope note:

This analysis identifies **gene-level, stroke-responsive endothelial surface/membrane-associated candidates**, not validated receptor proteins. The exported candidates require follow-up validation of isoform-specific topology, extracellular-domain exposure, receptor complex state, internalisation capacity, mouse-human translation, and experimental binding.

In the wider prototype, this notebook provides the first stage of the computational decision chain:

**transcriptomic mining → receptor prioritisation → peptide-design constraints → candidate generation → receptor-aware structure triage → experimental handoff**

## 1. Inspect downloaded raw files

The raw StrokeVis/GSE225948 files were downloaded into the local project directory. This first step lists the available compressed CSV files and previews the first rows of each table to understand their structure before selecting the relevant metadata and expression matrices.

In [41]:
# -------------------------
# 1.1 Imports and paths
# -------------------------

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/raw/GSE225948_RAW/")
files = sorted(data_dir.glob("*.csv.gz"))

print(f"Raw data directory: {data_dir}")
print(f"Number of .csv.gz files found: {len(files)}")

Raw data directory: ../data/raw/GSE225948_RAW
Number of .csv.gz files found: 58


In [42]:
# -------------------------
# 1.2 List available files
# -------------------------

for file in files:
    print("-", file.name)

- GSM7060815_Brain_GR180716_counts.csv.gz
- GSM7060815_Brain_GR180716_metadata.csv.gz
- GSM7060816_Brain_GR181128_counts.csv.gz
- GSM7060816_Brain_GR181128_metadata.csv.gz
- GSM7060817_Brain_GR181212_counts.csv.gz
- GSM7060817_Brain_GR181212_metadata.csv.gz
- GSM7060818_Brain_GR190110_counts.csv.gz
- GSM7060818_Brain_GR190110_metadata.csv.gz
- GSM7060819_Brain_GR180426_counts.csv.gz
- GSM7060819_Brain_GR180426_metadata.csv.gz
- GSM7060820_Brain_GR180614_counts.csv.gz
- GSM7060820_Brain_GR180614_metadata.csv.gz
- GSM7060821_Brain_GR180919_counts.csv.gz
- GSM7060821_Brain_GR180919_metadata.csv.gz
- GSM7060822_Brain_GR181024_counts.csv.gz
- GSM7060822_Brain_GR181024_metadata.csv.gz
- GSM7060823_Brain_GR180125_counts.csv.gz
- GSM7060823_Brain_GR180125_metadata.csv.gz
- GSM7060824_Brain_GR180613_counts.csv.gz
- GSM7060824_Brain_GR180613_metadata.csv.gz
- GSM7060825_Brain_GR180905_counts.csv.gz
- GSM7060825_Brain_GR180905_metadata.csv.gz
- GSM7060826_Brain_GR181114_counts.csv.gz
- GSM7060826

In [43]:
# -------------------------
# 1.3 Preview the first files
# -------------------------

for file in files[:5]:
    print("\n" + "=" * 80)
    print(file.name)

    df_preview = pd.read_csv(file, nrows=5)

    print("Preview shape:", df_preview.shape)
    display(df_preview)


GSM7060815_Brain_GR180716_counts.csv.gz
Preview shape: (5, 3466)


,Unnamed: 0,BRS02R1GGGATCGATATG,BRS02R1ATTCAATATCAC,BRS02R1ATTTCCGTCCGC,BRS02R1TAGCGCGAGACC,BRS02R1GCTGATTATTTG,BRS02R1CTATCTCTCGAA,BRS02R1CTGAATTTCCCC,BRS02R1ATTCACACACTT,BRS02R1ATTTGCGAGTGA,...,BRS02R1CATCTGTCGGAC,BRS02R1ACGGGGTTAATG,BRS02R1CACTACTTGTCC,BRS02R1AGTCGGCTATAG,BRS02R1AGAGCACTCGGC,BRS02R1GGTCGCGTTGAT,BRS02R1GGCGTTGGTACC,BRS02R1TGTTCCGAGAAG,BRS02R1GTGATGAAAAGT,BRS02R1TTTTTTTTTCGN
0,0610009B22Rik,0,0,0,1.992199,0,0.97048,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0610009E02Rik,0,0,0,0.000000,0,0.00000,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0610009L18Rik,0,0,0,0.000000,0,0.00000,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0610010F05Rik,0,0,0,0.000000,0,0.00000,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0610010K14Rik,0,0,0,0.000000,0,0.00000,0,0,0,...,0,0,0,0,0,0,0,0,0,0



GSM7060815_Brain_GR180716_metadata.csv.gz
Preview shape: (5, 15)


,Unnamed: 0,study.ident,nCount_RNA,nFeature_RNA,percent.mt,tissue,organism,strain,sex,age,treatment,Replicate,Sample_description,sub.celltype,parent
0,BRS02R1GGGATCGATATG,GR180716,2988.583761,973,5.347042,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Epi,Epi
1,BRS02R1ATTCAATATCAC,GR180716,3825.015376,1138,0.340638,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Gran5,Gran
2,BRS02R1ATTTCCGTCCGC,GR180716,3426.051680,1915,0.302520,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,BAM2,BAM
3,BRS02R1TAGCGCGAGACC,GR180716,3953.069361,1494,1.176273,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Epi,Epi
4,BRS02R1GCTGATTATTTG,GR180716,2841.509538,1526,0.655353,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,MdC3,MdC



GSM7060816_Brain_GR181128_counts.csv.gz
Preview shape: (5, 4715)


,Unnamed: 0,BRS02R2TCACGGTTCGTA,BRS02R2ATAAGGCGGCAA,BRS02R2TACAAAACCGAA,BRS02R2ACCGGATCGGAC,BRS02R2AGCAAATTCTCT,BRS02R2ACCCCCAGAGTT,BRS02R2TCTAATCTCATC,BRS02R2CCGACCCGAGAA,BRS02R2GTTCAGCAATTA,...,BRS02R2ATACGTGGCCCC,BRS02R2CGGGGTGAGGGG,BRS02R2GGAAGGCTACAG,BRS02R2GTGTGTTTTATC,BRS02R2ATCGACAGGGGG,BRS02R2GGGTGGGGTAGA,BRS02R2AATATTGGCTCT,BRS02R2AGTCGCGTCTGA,BRS02R2ACTGCTTCATAA,BRS02R2ACTGCTTACTGA
0,0610009B22Rik,0.000000,0,0,0,0,0,0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
1,0610009E02Rik,0.000000,0,0,0,0,0,0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
2,0610009L18Rik,0.000000,0,0,0,0,0,0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
3,0610010F05Rik,0.000000,0,0,0,0,0,0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
4,0610010K14Rik,0.749681,0,0,0,0,0,0,0.986626,0,...,0,0,0,0,0,0,0,0,0,0



GSM7060816_Brain_GR181128_metadata.csv.gz
Preview shape: (5, 15)


,Unnamed: 0,study.ident,nCount_RNA,nFeature_RNA,percent.mt,tissue,organism,strain,sex,age,treatment,Replicate,Sample_description,sub.celltype,parent
0,BRS02R2TCACGGTTCGTA,GR181128,3018.552883,2123,3.494441,brain,mus musculus,C57BL/6J,male,W8,Sham,2,2d sham; hemisphere; CD45hi + MG + EC,Mg2,Mg
1,BRS02R2ATAAGGCGGCAA,GR181128,2808.690684,1834,3.624112,brain,mus musculus,C57BL/6J,male,W8,Sham,2,2d sham; hemisphere; CD45hi + MG + EC,Mg2,Mg
2,BRS02R2TACAAAACCGAA,GR181128,2513.732804,1577,2.662910,brain,mus musculus,C57BL/6J,male,W8,Sham,2,2d sham; hemisphere; CD45hi + MG + EC,Mg2,Mg
3,BRS02R2ACCGGATCGGAC,GR181128,2822.869808,1553,3.014790,brain,mus musculus,C57BL/6J,male,W8,Sham,2,2d sham; hemisphere; CD45hi + MG + EC,Mg2,Mg
4,BRS02R2AGCAAATTCTCT,GR181128,2896.671892,805,0.946141,brain,mus musculus,C57BL/6J,male,W8,Sham,2,2d sham; hemisphere; CD45hi + MG + EC,Gran5,Gran



GSM7060817_Brain_GR181212_counts.csv.gz
Preview shape: (5, 3049)


,Unnamed: 0,BRS02R3TTTATTTCGGTC,BRS02R3CAAACTTTGCTG,BRS02R3GCGAAGATCACA,BRS02R3CGATCACGATCG,BRS02R3CTATGTTGTTTT,BRS02R3CAGTTGCTCACT,BRS02R3TGTACTTGAATC,BRS02R3ACAATGCACGCC,BRS02R3TATCATGGCCCG,...,BRS02R3CCGCGGCGGTGC,BRS02R3TGGGCCGGTATC,BRS02R3GTCTCCCTCTAT,BRS02R3CGCCTGGTAGTG,BRS02R3ATGAATTTCTTT,BRS02R3ATACACAGGTAC,BRS02R3TTCCCCTGCTTG,BRS02R3AGATAGTCACCG,BRS02R3GTGGGCGGCCTG,BRS02R3CTCCCCTGCTTA
0,0610009B22Rik,0,0,0,0,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000
1,0610009E02Rik,0,0,0,0,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.981743
2,0610009L18Rik,0,0,0,0,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000
3,0610010F05Rik,0,0,0,0,0.80012,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000
4,0610010K14Rik,0,0,0,0,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000


## 2. Combine metadata across sequencing samples

Each downloaded sample includes a metadata table describing cell-level annotations such as tissue, treatment, sample description, parent cell type, and sub-cell type. These metadata files are concatenated into one table so that endothelial cells and stroke-related experimental groups can be selected consistently across samples.

In [44]:
# -------------------------
# 2.1 Load and concatenate metadata files
# -------------------------

metadata_files = sorted(data_dir.glob("*metadata.csv.gz"))

print(f"Metadata files found: {len(metadata_files)}")
for f in metadata_files:
    print("-", f.name)

all_meta = []

for f in metadata_files:
    meta = pd.read_csv(f)
    meta["source_file"] = f.name
    all_meta.append(meta)

meta_all = pd.concat(all_meta, ignore_index=True)

print("Combined metadata shape:", meta_all.shape)
display(meta_all.head())

Metadata files found: 29
- GSM7060815_Brain_GR180716_metadata.csv.gz
- GSM7060816_Brain_GR181128_metadata.csv.gz
- GSM7060817_Brain_GR181212_metadata.csv.gz
- GSM7060818_Brain_GR190110_metadata.csv.gz
- GSM7060819_Brain_GR180426_metadata.csv.gz
- GSM7060820_Brain_GR180614_metadata.csv.gz
- GSM7060821_Brain_GR180919_metadata.csv.gz
- GSM7060822_Brain_GR181024_metadata.csv.gz
- GSM7060823_Brain_GR180125_metadata.csv.gz
- GSM7060824_Brain_GR180613_metadata.csv.gz
- GSM7060825_Brain_GR180905_metadata.csv.gz
- GSM7060826_Brain_GR181114_metadata.csv.gz
- GSM7060827_Brain_aged_GR210708_metadata.csv.gz
- GSM7060828_Brain_aged_GR200728_metadata.csv.gz
- GSM7060829_Brain_aged_GR200723_metadata.csv.gz
- GSM7060830_Brain_aged_GR200716_metadata.csv.gz
- GSM7060831_Brain_aged_GR210225_metadata.csv.gz
- GSM7060832_Brain_aged_GR200812_metadata.csv.gz
- GSM7060833_Blood_GR18110701_metadata.csv.gz
- GSM7060834_Blood_GR18121901_metadata.csv.gz
- GSM7060835_Blood_GR190417_metadata.csv.gz
- GSM7060836_Bloo

,Unnamed: 0,study.ident,nCount_RNA,nFeature_RNA,percent.mt,tissue,organism,strain,sex,age,treatment,Replicate,Sample_description,sub.celltype,parent,source_file
0,BRS02R1GGGATCGATATG,GR180716,2988.583761,973,5.347042,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Epi,Epi,GSM7060815_Brain_GR180716_metadata.csv.gz
1,BRS02R1ATTCAATATCAC,GR180716,3825.015376,1138,0.340638,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Gran5,Gran,GSM7060815_Brain_GR180716_metadata.csv.gz
2,BRS02R1ATTTCCGTCCGC,GR180716,3426.051680,1915,0.302520,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,BAM2,BAM,GSM7060815_Brain_GR180716_metadata.csv.gz
3,BRS02R1TAGCGCGAGACC,GR180716,3953.069361,1494,1.176273,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,Epi,Epi,GSM7060815_Brain_GR180716_metadata.csv.gz
4,BRS02R1GCTGATTATTTG,GR180716,2841.509538,1526,0.655353,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,MdC3,MdC,GSM7060815_Brain_GR180716_metadata.csv.gz


In [45]:
# -------------------------
# 2.2 Inspect metadata columns
# -------------------------

print("Metadata columns:")
for col in meta_all.columns:
    print("-", col)

Metadata columns:
- Unnamed: 0
- study.ident
- nCount_RNA
- nFeature_RNA
- percent.mt
- tissue
- organism
- strain
- sex
- age
- treatment
- Replicate
- Sample_description
- sub.celltype
- parent
- source_file


In [46]:
# -------------------------
# 2.3 Inspect tissue and treatment labels
# -------------------------

print("Tissues:")
display(meta_all["tissue"].value_counts().to_frame("n_cells"))

print("Treatments:")
display(meta_all["treatment"].value_counts().to_frame("n_cells"))

print("Sample descriptions:")
display(meta_all["Sample_description"].value_counts().to_frame("n_cells"))

Tissues:


,n_cells
tissue,
brain,55268
PB,51709


Treatments:


,n_cells
treatment,
D02,38311
Sham,36970
D14,31696


Sample descriptions:


,n_cells
Sample_description,
Peripheral blood leukocytes (240K),51709
2d sham; hemisphere; CD45hi + MG + EC,15029
2d stroke; ipsilateral hemisphere; CD45hi + MG + EC,14823
14d stroke; ipsilateral hemisphere; CD45hi + MG + EC,13417
2d stroke; ipsilateral hemisphere; CD45hi MG + EC,3682
2d sham; ipsilateral hemisphere; CD45hi MG + EC,3069
14d stroke; ipsilateral hemisphere; CD45hi MG + EC,2636
EC (20K); MG(20K); CD45hi(120k);,1469
EC (54K); MG(52K); CD45hi(27k);,1143


In [47]:
# -------------------------
# 2.4 Inspect cell-type annotations
# -------------------------

print("Parent cell types:")
display(meta_all["parent"].value_counts().to_frame("n_cells"))

print("Top sub-cell types:")
display(meta_all["sub.celltype"].value_counts().head(50).to_frame("n_cells"))

Parent cell types:


,n_cells
parent,
Mg,24707
Bc,18881
Neu,17186
MdC,12553
Tc,9047
EC,7627
Mo,6558
DC,4256
NK,2077


Top sub-cell types:


,n_cells
sub.celltype,
Bc1,8910
Mg1,8519
Neu1,5918
Mg2,5567
Mg3,5118
Neu2,4698
Bc2,3589
Tc1,3257
Tc2,3203


## 3. Select brain endothelial cells

The receptor-mining workflow focuses on endothelial cells from brain tissue because these cells represent the blood-brain barrier and vascular interface relevant to peptide-shuttle design. This section filters the combined metadata to brain endothelial cells and checks whether enough cells are available across treatments, subclusters, and source files.

In [48]:
# -------------------------
# 3.1 Filter brain endothelial cells
# -------------------------

ec_meta = meta_all[
    (meta_all["tissue"] == "brain")
    & (meta_all["parent"] == "EC")
].copy()

print("Brain endothelial metadata shape:", ec_meta.shape)
display(ec_meta.head())

Brain endothelial metadata shape: (7627, 16)


,Unnamed: 0,study.ident,nCount_RNA,nFeature_RNA,percent.mt,tissue,organism,strain,sex,age,treatment,Replicate,Sample_description,sub.celltype,parent,source_file
14,BRS02R1CACAGCCGGTTC,GR180716,2994.936921,1594,1.055879,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,EC3,EC,GSM7060815_Brain_GR180716_metadata.csv.gz
23,BRS02R1TCTCCGGGCTCG,GR180716,2973.772621,1563,1.198053,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,EC3,EC,GSM7060815_Brain_GR180716_metadata.csv.gz
37,BRS02R1GCCTTCTCTGAC,GR180716,2167.838957,1154,0.560680,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,EC1,EC,GSM7060815_Brain_GR180716_metadata.csv.gz
46,BRS02R1GACTACGTTTTT,GR180716,1476.342542,1027,0.560441,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,EC4,EC,GSM7060815_Brain_GR180716_metadata.csv.gz
58,BRS02R1TAGGCGAGGATT,GR180716,1813.946297,1253,0.635394,brain,mus musculus,C57BL/6J,male,W8,Sham,1,2d sham; hemisphere; CD45hi + MG + EC,EC1,EC,GSM7060815_Brain_GR180716_metadata.csv.gz


In [49]:
# -------------------------
# 3.2 Inspect endothelial cells by treatment
# -------------------------

display(
    ec_meta["treatment"]
    .value_counts()
    .to_frame("n_cells")
)

,n_cells
treatment,
D02,3239
Sham,2602
D14,1786


In [50]:
# -------------------------
# 3.3 Inspect endothelial subclusters
# -------------------------

display(
    ec_meta["sub.celltype"]
    .value_counts()
    .to_frame("n_cells")
)

,n_cells
sub.celltype,
EC1,3126
EC2,1621
EC3,950
EC4,768
EC5,526
EC6,350
EC7,151
EC8,105
EC9,30


In [51]:
# -------------------------
# 3.4 Cross-tabulate endothelial subclusters by treatment
# -------------------------

ec_subcluster_treatment_counts = pd.crosstab(
    ec_meta["sub.celltype"],
    ec_meta["treatment"],
)

display(ec_subcluster_treatment_counts)

treatment,D02,D14,Sham
sub.celltype,,,
EC1,774,788,1564
EC2,1486,57,78
EC3,269,452,229
EC4,244,123,401
EC5,202,233,91
EC6,154,74,122
EC7,61,30,60
EC8,29,28,48
EC9,20,1,9


In [52]:
# -------------------------
# 3.5 Inspect endothelial cells by treatment and source file
# -------------------------

ec_by_source = (
    ec_meta
    .groupby(["treatment", "source_file"])
    .size()
    .reset_index(name="n_cells")
    .sort_values(["treatment", "source_file"])
)

display(ec_by_source)

,treatment,source_file,n_cells
0,D02,GSM7060819_Brain_GR180426_metadata.csv.gz,360
1,D02,GSM7060820_Brain_GR180614_metadata.csv.gz,458
2,D02,GSM7060821_Brain_GR180919_metadata.csv.gz,502
3,D02,GSM7060822_Brain_GR181024_metadata.csv.gz,1171
4,D02,GSM7060829_Brain_aged_GR200723_metadata.csv.gz,325
5,D02,GSM7060830_Brain_aged_GR200716_metadata.csv.gz,219
6,D02,GSM7060831_Brain_aged_GR210225_metadata.csv.gz,204
7,D14,GSM7060823_Brain_GR180125_metadata.csv.gz,674
8,D14,GSM7060824_Brain_GR180613_metadata.csv.gz,319
9,D14,GSM7060825_Brain_GR180905_metadata.csv.gz,300


## 4. Validate endothelial identity using marker genes

Before prioritising receptor candidates, the endothelial-cell annotation is checked using known endothelial and blood-brain-barrier markers. The expectation is that brain EC-labelled cells should express endothelial markers such as `Cldn5`, `Pecam1`, `Cdh5`, `Kdr`, `Flt1`, `Tek`, `Slc2a1`, and `Mfsd2a`, while non-endothelial markers such as immune, pericyte, mural, and astrocyte markers should remain comparatively low.

This step is a sanity check to support that the EC subset used for downstream receptor mining is biologically reasonable.

In [53]:
# -------------------------
# 4.1 Define endothelial and negative-control marker genes
# -------------------------

endothelial_markers = [
    "Cldn5", "Pecam1", "Cdh5", "Kdr", "Flt1", "Tek",
    "Slc2a1", "Mfsd2a", "Abcb1a", "Abcg2", "Tfrc", "Slc7a5",
    "Vwf", "Cav1", "Plvap",
]

negative_markers = [
    "Ptprc",   # immune cells
    "Aif1",   # microglia/macrophages
    "Pdgfrb", # pericytes
    "Rgs5",   # mural/pericyte
    "Gfap",   # astrocytes
    "Aqp4",   # astrocyte/endfoot-associated
]

marker_genes = endothelial_markers + negative_markers

print("Endothelial markers:", len(endothelial_markers))
print("Negative-control markers:", len(negative_markers))
print("Total marker genes:", len(marker_genes))

Endothelial markers: 15
Negative-control markers: 6
Total marker genes: 21


In [54]:
# -------------------------
# 4.2 Helper function to calculate EC marker expression per sample
# -------------------------

def load_gene_panel_for_ec(meta_path, genes):
    counts_path = Path(str(meta_path).replace("_metadata.csv.gz", "_counts.csv.gz"))

    if not counts_path.exists():
        print(f"Counts file missing for: {meta_path.name}")
        return None

    meta = pd.read_csv(meta_path)
    meta["Unnamed: 0"] = meta["Unnamed: 0"].astype(str)

    ec_meta_sample = meta[
        (meta["tissue"] == "brain")
        & (meta["parent"] == "EC")
    ].copy()

    if ec_meta_sample.empty:
        return None

    ec_cells = ec_meta_sample["Unnamed: 0"].tolist()

    counts = pd.read_csv(counts_path, index_col=0)
    counts.index = counts.index.astype(str)
    counts.columns = counts.columns.astype(str)

    available_genes = [g for g in genes if g in counts.index]
    available_cells = [c for c in ec_cells if c in counts.columns]

    print(
        meta_path.name,
        "| EC cells:", len(ec_cells),
        "| matched cells:", len(available_cells),
        "| matched genes:", len(available_genes),
    )

    if len(available_genes) == 0 or len(available_cells) == 0:
        return None

    expr = counts.loc[available_genes, available_cells]

    out = expr.mean(axis=1).reset_index()
    out.columns = ["gene", "mean_expression"]

    out["treatment"] = ec_meta_sample["treatment"].iloc[0]
    out["source_file"] = meta_path.name
    out["n_ec_cells"] = len(available_cells)

    return out

In [55]:
# -------------------------
# 4.3 Calculate marker expression across EC samples
# -------------------------

all_marker_expr = []

for meta_path in sorted(data_dir.glob("*metadata.csv.gz")):
    result = load_gene_panel_for_ec(meta_path, marker_genes)
    if result is not None:
        all_marker_expr.append(result)

if len(all_marker_expr) == 0:
    raise ValueError("No marker expression results found. Check data_dir and filenames.")

marker_expr = pd.concat(all_marker_expr, ignore_index=True)

print("Marker expression table shape:", marker_expr.shape)
display(marker_expr.head())

GSM7060815_Brain_GR180716_metadata.csv.gz | EC cells: 546 | matched cells: 546 | matched genes: 20
GSM7060816_Brain_GR181128_metadata.csv.gz | EC cells: 617 | matched cells: 617 | matched genes: 20
GSM7060817_Brain_GR181212_metadata.csv.gz | EC cells: 321 | matched cells: 321 | matched genes: 20
GSM7060818_Brain_GR190110_metadata.csv.gz | EC cells: 295 | matched cells: 295 | matched genes: 20
GSM7060819_Brain_GR180426_metadata.csv.gz | EC cells: 360 | matched cells: 360 | matched genes: 20
GSM7060820_Brain_GR180614_metadata.csv.gz | EC cells: 458 | matched cells: 458 | matched genes: 20
GSM7060821_Brain_GR180919_metadata.csv.gz | EC cells: 502 | matched cells: 502 | matched genes: 20
GSM7060822_Brain_GR181024_metadata.csv.gz | EC cells: 1171 | matched cells: 1171 | matched genes: 20
GSM7060823_Brain_GR180125_metadata.csv.gz | EC cells: 674 | matched cells: 674 | matched genes: 20
GSM7060824_Brain_GR180613_metadata.csv.gz | EC cells: 319 | matched cells: 319 | matched genes: 20
GSM70608

,gene,mean_expression,treatment,source_file,n_ec_cells
0,Cldn5,4.046015,Sham,GSM7060815_Brain_GR180716_metadata.csv.gz,546
1,Pecam1,0.668399,Sham,GSM7060815_Brain_GR180716_metadata.csv.gz,546
2,Cdh5,0.363003,Sham,GSM7060815_Brain_GR180716_metadata.csv.gz,546
3,Kdr,0.250754,Sham,GSM7060815_Brain_GR180716_metadata.csv.gz,546
4,Flt1,3.538528,Sham,GSM7060815_Brain_GR180716_metadata.csv.gz,546


In [56]:
# -------------------------
# 4.4 Summarise marker expression by treatment
# -------------------------

marker_summary = (
    marker_expr
    .groupby(["gene", "treatment"])["mean_expression"]
    .mean()
    .reset_index()
    .pivot(index="gene", columns="treatment", values="mean_expression")
)

display(marker_summary)

treatment,D02,D14,Sham
gene,,,
Abcb1a,7.003937e-01,1.268677e+00,1.054301
Abcg2,5.175889e-01,5.456159e-01,0.431363
Aif1,1.219461e-15,4.575544e-11,0.003254
Aqp4,0.000000e+00,0.000000e+00,0.000000
Cav1,2.438081e-01,2.362453e-01,0.174210
Cdh5,2.364959e-01,1.750940e-01,0.129798
Cldn5,2.277336e+00,2.905112e+00,1.813087
Flt1,1.591764e+00,2.360281e+00,2.061095
Gfap,4.220719e-04,0.000000e+00,0.000000


In [57]:
# -------------------------
# 4.5 Save EC marker validation outputs
# -------------------------

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

marker_summary_path = processed_dir / "ec_marker_summary.csv"
marker_expr_path = processed_dir / "ec_marker_expression_by_sample.csv"

marker_summary.to_csv(marker_summary_path)
marker_expr.to_csv(marker_expr_path, index=False)

print("Saved marker summary:", marker_summary_path)
print("Saved marker expression by sample:", marker_expr_path)

Saved marker summary: ../data/processed/ec_marker_summary.csv
Saved marker expression by sample: ../data/processed/ec_marker_expression_by_sample.csv


## 5. Build EC pseudobulk expression and compare stroke time points

To prioritise stroke-induced endothelial receptor candidates, a sample-level EC pseudobulk matrix is created. For each sample, the expression of each gene is averaged across brain endothelial cells. This converts the cell-level count matrices into one expression profile per biological sample.

The pseudobulk matrix is then used to compare early and subacute stroke time points against sham controls:

- `D02` versus `Sham`
- `D14` versus `Sham`

The resulting tables provide simple effect-size, log2 fold-change proxy, and Welch t-test statistics for each gene. These outputs are used later for receptor prioritisation rather than formal differential-expression claims.

In [58]:
# -------------------------
# 5.1 Build EC pseudobulk expression matrix
# -------------------------

from scipy.stats import ttest_ind

out_dir = Path("../data/processed/")
out_dir.mkdir(parents=True, exist_ok=True)

sample_expr = []
sample_info = []

for meta_path in sorted(data_dir.glob("*metadata.csv.gz")):
    counts_path = Path(str(meta_path).replace("_metadata.csv.gz", "_counts.csv.gz"))

    if not counts_path.exists():
        continue

    meta = pd.read_csv(meta_path)
    meta["Unnamed: 0"] = meta["Unnamed: 0"].astype(str)

    # Keep only brain endothelial cells
    ec_meta_sample = meta[
        (meta["tissue"] == "brain")
        & (meta["parent"] == "EC")
    ].copy()

    if ec_meta_sample.empty:
        continue

    ec_cells = ec_meta_sample["Unnamed: 0"].tolist()

    counts = pd.read_csv(counts_path, index_col=0)
    counts.index = counts.index.astype(str)
    counts.columns = counts.columns.astype(str)

    available_cells = [c for c in ec_cells if c in counts.columns]

    if len(available_cells) == 0:
        continue

    # Pseudobulk: average expression of each gene across EC cells in this sample
    mean_expr = counts[available_cells].mean(axis=1)

    sample_id = meta_path.name.replace("_metadata.csv.gz", "")
    mean_expr.name = sample_id

    sample_expr.append(mean_expr)

    sample_info.append({
        "sample_id": sample_id,
        "source_file": meta_path.name,
        "treatment": ec_meta_sample["treatment"].iloc[0],
        "age": ec_meta_sample["age"].iloc[0],
        "sex": ec_meta_sample["sex"].iloc[0],
        "n_ec_cells": len(available_cells),
    })

ec_pseudobulk = pd.concat(sample_expr, axis=1)
ec_sample_info = pd.DataFrame(sample_info)

print("Pseudobulk matrix shape:", ec_pseudobulk.shape)
display(ec_sample_info)

Pseudobulk matrix shape: (27641, 18)


,sample_id,source_file,treatment,age,sex,n_ec_cells
0,GSM7060815_Brain_GR180716,GSM7060815_Brain_GR180716_metadata.csv.gz,Sham,W8,male,546
1,GSM7060816_Brain_GR181128,GSM7060816_Brain_GR181128_metadata.csv.gz,Sham,W8,male,617
2,GSM7060817_Brain_GR181212,GSM7060817_Brain_GR181212_metadata.csv.gz,Sham,W10,male,321
3,GSM7060818_Brain_GR190110,GSM7060818_Brain_GR190110_metadata.csv.gz,Sham,W8,male,295
4,GSM7060819_Brain_GR180426,GSM7060819_Brain_GR180426_metadata.csv.gz,D02,W8,male,360
5,GSM7060820_Brain_GR180614,GSM7060820_Brain_GR180614_metadata.csv.gz,D02,W8,male,458
6,GSM7060821_Brain_GR180919,GSM7060821_Brain_GR180919_metadata.csv.gz,D02,W8,male,502
7,GSM7060822_Brain_GR181024,GSM7060822_Brain_GR181024_metadata.csv.gz,D02,W8,male,1171
8,GSM7060823_Brain_GR180125,GSM7060823_Brain_GR180125_metadata.csv.gz,D14,W10,male,674
9,GSM7060824_Brain_GR180613,GSM7060824_Brain_GR180613_metadata.csv.gz,D14,W8,male,319


In [59]:
# -------------------------
# 5.2 Save EC pseudobulk outputs
# -------------------------

ec_pseudobulk_path = out_dir / "ec_pseudobulk_mean_expression.csv"
ec_sample_info_path = out_dir / "ec_sample_info.csv"

ec_pseudobulk.to_csv(ec_pseudobulk_path)
ec_sample_info.to_csv(ec_sample_info_path, index=False)

print("Saved EC pseudobulk matrix:", ec_pseudobulk_path)
print("Saved EC sample information:", ec_sample_info_path)

Saved EC pseudobulk matrix: ../data/processed/ec_pseudobulk_mean_expression.csv
Saved EC sample information: ../data/processed/ec_sample_info.csv


In [60]:
# -------------------------
# 5.3 Define treatment comparison helper
# -------------------------

def compare_treatments(expr, info, case, control="Sham"):
    case_samples = info.loc[info["treatment"] == case, "sample_id"].tolist()
    control_samples = info.loc[info["treatment"] == control, "sample_id"].tolist()

    print(f"{case} samples:", len(case_samples))
    print(f"{control} samples:", len(control_samples))

    if len(case_samples) == 0 or len(control_samples) == 0:
        raise ValueError(f"Missing samples for comparison: {case} vs {control}")

    x_case = expr[case_samples]
    x_control = expr[control_samples]

    mean_case = x_case.mean(axis=1)
    mean_control = x_control.mean(axis=1)

    effect = mean_case - mean_control

    eps = 1e-6
    log2fc_proxy = np.log2((mean_case + eps) / (mean_control + eps))

    pvals = ttest_ind(
        x_case.T,
        x_control.T,
        axis=0,
        equal_var=False,
        nan_policy="omit",
    ).pvalue

    out = pd.DataFrame({
        "gene": expr.index,
        f"mean_{case}": mean_case.values,
        f"mean_{control}": mean_control.values,
        f"effect_{case}_vs_{control}": effect.values,
        f"log2fc_proxy_{case}_vs_{control}": log2fc_proxy.values,
        f"pvalue_{case}_vs_{control}": pvals,
    })

    return out.sort_values(
        f"effect_{case}_vs_{control}",
        ascending=False,
    ).reset_index(drop=True)

In [61]:
# -------------------------
# 5.4 Compare D02 and D14 versus Sham
# -------------------------

de_d02 = compare_treatments(
    ec_pseudobulk,
    ec_sample_info,
    case="D02",
)

de_d14 = compare_treatments(
    ec_pseudobulk,
    ec_sample_info,
    case="D14",
)

de_d02_path = out_dir / "ec_DE_D02_vs_Sham.csv"
de_d14_path = out_dir / "ec_DE_D14_vs_Sham.csv"

de_d02.to_csv(de_d02_path, index=False)
de_d14.to_csv(de_d14_path, index=False)

print("Saved D02 comparison:", de_d02_path)
print("Saved D14 comparison:", de_d14_path)

print("Top D02-induced EC genes:")
display(de_d02.head(20))

print("Top D14-induced EC genes:")
display(de_d14.head(20))

D02 samples: 7
Sham samples: 6
D14 samples: 5
Sham samples: 6
Saved D02 comparison: ../data/processed/ec_DE_D02_vs_Sham.csv
Saved D14 comparison: ../data/processed/ec_DE_D14_vs_Sham.csv
Top D02-induced EC genes:


,gene,mean_D02,mean_Sham,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,pvalue_D02_vs_Sham
0,Tmsb4x,8.151992,2.224929,5.927064,1.873393,0.001179
1,Igfbp7,6.496007,1.453066,5.042942,2.160453,0.008618
2,Ly6a,6.753810,2.183473,4.570337,1.629077,0.008333
3,Ctla2a,4.136150,0.477916,3.658234,3.113456,0.005249
4,Sparc,4.324855,0.831155,3.493701,2.379462,0.021555
5,mt-Rnr2,22.573062,19.394774,3.178288,0.218934,0.656604
6,Actb,4.920596,1.761817,3.158780,1.481769,0.030027
7,Rplp1,3.151766,0.759192,2.392574,2.053622,0.006898
8,Fth1,3.664283,1.307899,2.356383,1.486278,0.002023
9,Malat1,5.356480,3.006139,2.350341,0.833373,0.136055


Top D14-induced EC genes:


,gene,mean_D14,mean_Sham,effect_D14_vs_Sham,log2fc_proxy_D14_vs_Sham,pvalue_D14_vs_Sham
0,Bsg,8.237078,5.933134,2.303945,0.473338,0.319651
1,Malat1,4.893720,3.006139,1.887581,0.703020,0.217132
2,Tmsb4x,3.805056,2.224929,1.580128,0.774158,0.151303
3,Hbb-bs,1.440309,0.176782,1.263527,3.026329,0.130952
4,Itm2a,2.889887,1.651506,1.238381,0.807231,0.235502
5,Pltp,2.359225,1.130960,1.228264,1.060764,0.141996
6,B2m,2.226214,1.035883,1.190331,1.103731,0.090017
7,Cldn5,2.905112,1.813087,1.092024,0.680145,0.279994
8,Actb,2.770267,1.761817,1.008450,0.652961,0.185234
9,Hsp90ab1,2.101411,1.262513,0.838899,0.735060,0.176462


## 6. Inspect known BBB receptor, transporter, and endothelial surface candidates

Before mining the full transcriptome for novel receptor candidates, a curated target panel is inspected. This panel includes known or plausible BBB transport receptors, endothelial receptors, barrier identity markers, efflux transporters, inflammatory adhesion molecules, and stroke-induced surface candidates.

This step serves as a biological sanity check. It asks whether known BBB/endothelial genes are detected in the EC pseudobulk table and whether they are preserved, induced, or suppressed after stroke. The resulting table is used for interpretation and context, while the final receptor shortlist is generated in later sections using broader surface-accessibility and induction criteria.

In [62]:
# -------------------------
# 6.1 Define curated BBB/receptor target panel
# -------------------------

target_panel = {
    # known / plausible BBB shuttle or transport-relevant candidates
    "Tfrc": "known BBB receptor-mediated uptake candidate",
    "Lrp1": "endocytic BBB transport receptor",
    "Insr": "receptor-mediated transcytosis precedent",
    "Igf1r": "receptor-mediated transcytosis precedent",
    "Bsg": "membrane glycoprotein, vascular/inflammatory relevance",
    "Cd36": "scavenger receptor, vascular/endothelial relevance",
    "Scarb1": "scavenger receptor, lipid transport relevance",

    # BBB transporters / barrier identity
    "Slc2a1": "BBB glucose transporter and EC identity marker",
    "Slc7a5": "BBB amino-acid transporter",
    "Mfsd2a": "BBB lipid transporter and barrier identity marker",
    "Abcb1a": "BBB efflux transporter",
    "Abcg2": "BBB efflux transporter",
    "Slco1a4": "brain endothelial solute carrier candidate",

    # vascular / endothelial receptors
    "Kdr": "VEGF receptor, endothelial receptor",
    "Flt1": "VEGF receptor, endothelial receptor",
    "Tek": "Tie2 endothelial receptor",
    "Eng": "endothelial accessory receptor",

    # stroke/inflammatory endothelial activation
    "Icam1": "activated endothelial adhesion receptor",
    "Vcam1": "activated endothelial adhesion receptor",
    "Sele": "activated endothelial adhesion receptor",
    "Plvap": "vascular permeability-associated membrane protein",
    "Ly6a": "stroke-induced endothelial surface marker candidate",
    "Ly6c1": "stroke-associated vascular/immune interface marker",

    # membrane/cell-surface candidates from top results
    "Igfbp7": "secreted/perivascular EC-associated factor, target-context marker",
    "Itm2a": "membrane protein candidate",
}

print("Number of curated targets:", len(target_panel))

Number of curated targets: 25


In [63]:
# -------------------------
# 6.2 Define curated target-table helper
# -------------------------

def make_target_table(de_d02, de_d14, target_panel):
    d02_cols = [
        "gene",
        "mean_D02",
        "mean_Sham",
        "effect_D02_vs_Sham",
        "log2fc_proxy_D02_vs_Sham",
        "pvalue_D02_vs_Sham",
    ]

    d14_cols = [
        "gene",
        "mean_D14",
        "mean_Sham",
        "effect_D14_vs_Sham",
        "log2fc_proxy_D14_vs_Sham",
        "pvalue_D14_vs_Sham",
    ]

    d02 = de_d02[d02_cols].copy()
    d14 = de_d14[d14_cols].copy()

    merged = d02.merge(
        d14,
        on="gene",
        how="outer",
        suffixes=("_D02table", "_D14table"),
    )

    panel = pd.DataFrame({
        "gene": list(target_panel.keys()),
        "BBB_relevance": list(target_panel.values()),
    })

    out = panel.merge(merged, on="gene", how="left")

    def classify_response(row):
        d02_effect = row.get("effect_D02_vs_Sham", 0)
        d14_effect = row.get("effect_D14_vs_Sham", 0)

        if pd.isna(d02_effect):
            d02_effect = 0
        if pd.isna(d14_effect):
            d14_effect = 0

        if d02_effect > 0.5 and d14_effect > 0.5:
            return "persistent stroke-induced"
        if d02_effect > 0.5 and d14_effect <= 0.5:
            return "early D02-induced"
        if d02_effect <= 0.5 and d14_effect > 0.5:
            return "late D14-induced"
        if d02_effect < -0.5 or d14_effect < -0.5:
            return "stroke-suppressed"
        return "preserved / weakly changed"

    out["stroke_response"] = out.apply(classify_response, axis=1)

    out["cell_source_evidence"] = (
        "brain EC pseudobulk; EC identity verified by canonical endothelial and BBB markers"
    )

    target_constraints = {
        "Tfrc": "bind extracellular receptor region; avoid disrupting iron transport",
        "Lrp1": "exploit endocytic uptake; avoid broad off-target uptake",
        "Insr": "avoid metabolic agonism while preserving transport",
        "Igf1r": "avoid growth-factor-like signalling activation",
        "Bsg": "target extracellular domain; check inflammatory/off-target expression",
        "Cd36": "avoid broad scavenger-receptor off-target effects",
        "Scarb1": "balance lipid-transport biology and endothelial specificity",
        "Slc2a1": "use mainly as BBB identity marker, not primary peptide target",
        "Slc7a5": "transporter biology relevant; peptide binding may be difficult",
        "Mfsd2a": "BBB identity marker; targetability requires caution",
        "Abcb1a": "efflux transporter; not ideal as shuttle target",
        "Abcg2": "efflux transporter; not ideal as shuttle target",
        "Slco1a4": "evaluate extracellular accessibility and species translation",
        "Kdr": "avoid angiogenic signalling activation",
        "Flt1": "avoid VEGF-pathway perturbation",
        "Tek": "avoid Tie2 pathway activation unless intended",
        "Eng": "evaluate vascular specificity and signalling risk",
        "Icam1": "stroke-activated targeting; avoid systemic immune adhesion effects",
        "Vcam1": "stroke-activated targeting; avoid broad inflammatory endothelium",
        "Sele": "stroke-activated targeting; likely inflammation-state specific",
        "Plvap": "vascular permeability marker; assess BBB specificity",
        "Ly6a": "strong mouse stroke EC signal; check human translational relevance",
        "Ly6c1": "stroke-associated but immune/vascular specificity must be checked",
        "Igfbp7": "use as EC/stroke context marker more than direct shuttle receptor",
        "Itm2a": "evaluate membrane topology and extracellular accessibility",
    }

    out["target_constraint"] = out["gene"].map(target_constraints)

    response_order = {
        "early D02-induced": 0,
        "persistent stroke-induced": 1,
        "late D14-induced": 2,
        "preserved / weakly changed": 3,
        "stroke-suppressed": 4,
    }

    out["response_order"] = out["stroke_response"].map(response_order)

    out = out.sort_values(
        ["response_order", "effect_D02_vs_Sham"],
        ascending=[True, False],
    ).drop(columns="response_order")

    return out.reset_index(drop=True)

In [64]:
# -------------------------
# 6.3 Build curated target table
# -------------------------

target_table = make_target_table(
    de_d02=de_d02,
    de_d14=de_d14,
    target_panel=target_panel,
)

display(target_table)

,gene,BBB_relevance,mean_D02,mean_Sham_D02table,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,pvalue_D02_vs_Sham,mean_D14,mean_Sham_D14table,effect_D14_vs_Sham,log2fc_proxy_D14_vs_Sham,pvalue_D14_vs_Sham,stroke_response,cell_source_evidence,target_constraint
0,Igfbp7,"secreted/perivascular EC-associated factor, ta...",6.496007,1.453066e+00,5.042942,2.160453,0.008618,1.609349,1.453066e+00,0.156283,0.147377,0.745711,early D02-induced,brain EC pseudobulk; EC identity verified by c...,use as EC/stroke context marker more than dire...
1,Ly6a,stroke-induced endothelial surface marker cand...,6.753810,2.183473e+00,4.570337,1.629077,0.008333,2.595743,2.183473e+00,0.412271,0.249523,0.602446,early D02-induced,brain EC pseudobulk; EC identity verified by c...,strong mouse stroke EC signal; check human tra...
2,Ly6c1,stroke-associated vascular/immune interface ma...,5.853538,3.717894e+00,2.135644,0.654823,0.123421,3.744299,3.717894e+00,0.026405,0.010210,0.981860,early D02-induced,brain EC pseudobulk; EC identity verified by c...,stroke-associated but immune/vascular specific...
3,Itm2a,membrane protein candidate,1.907739,1.651506e+00,0.256233,0.208081,0.641626,2.889887,1.651506e+00,1.238381,0.807231,0.235502,late D14-induced,brain EC pseudobulk; EC identity verified by c...,evaluate membrane topology and extracellular a...
4,Slco1a4,brain endothelial solute carrier candidate,1.008533,1.372543e+00,-0.364010,-0.444593,0.150921,1.997028,1.372543e+00,0.624485,0.541003,0.217050,late D14-induced,brain EC pseudobulk; EC identity verified by c...,evaluate extracellular accessibility and speci...
5,Bsg,"membrane glycoprotein, vascular/inflammatory r...",3.999826,5.933134e+00,-1.933308,-0.568857,0.159024,8.237078,5.933134e+00,2.303945,0.473338,0.319651,late D14-induced,brain EC pseudobulk; EC identity verified by c...,target extracellular domain; check inflammator...
6,Eng,endothelial accessory receptor,0.588987,2.291584e-01,0.359828,1.361886,0.020719,0.317102,2.291584e-01,0.087944,0.468600,0.349628,preserved / weakly changed,brain EC pseudobulk; EC identity verified by c...,evaluate vascular specificity and signalling risk
7,Slc2a1,BBB glucose transporter and EC identity marker,0.941885,6.223895e-01,0.319495,0.597732,0.207402,0.611308,6.223895e-01,-0.011081,-0.025918,0.956956,preserved / weakly changed,brain EC pseudobulk; EC identity verified by c...,"use mainly as BBB identity marker, not primary..."
8,Kdr,"VEGF receptor, endothelial receptor",0.330704,2.010880e-01,0.129616,0.717710,0.183229,0.334654,2.010880e-01,0.133566,0.734838,0.143415,preserved / weakly changed,brain EC pseudobulk; EC identity verified by c...,avoid angiogenic signalling activation
9,Abcg2,BBB efflux transporter,0.517589,4.313633e-01,0.086226,0.262903,0.230239,0.545616,4.313633e-01,0.114253,0.338981,0.379466,preserved / weakly changed,brain EC pseudobulk; EC identity verified by c...,efflux transporter; not ideal as shuttle target


In [65]:
# -------------------------
# 6.4 Save curated target table
# -------------------------

target_table_path = out_dir / "ec_receptor_transporter_target_table.csv"

target_table.to_csv(target_table_path, index=False)

print("Saved curated receptor/transporter target table:", target_table_path)

Saved curated receptor/transporter target table: ../data/processed/ec_receptor_transporter_target_table.csv


## 7. Identify data-driven stroke-responsive EC candidates

After inspecting a curated BBB/receptor panel, a broader data-driven screen is performed across the EC pseudobulk comparison results. The aim is to identify genes that are strongly induced after stroke and expressed in endothelial cells, before applying surface-accessibility and UniProt annotation filters in later sections.

This step removes obvious housekeeping or non-target-like genes, such as ribosomal, mitochondrial, haemoglobin, predicted, and microRNA genes. The remaining genes are filtered by minimum case expression and stroke-induced effect size.

In [66]:
# -------------------------
# 7.1 Define data-driven candidate filter
# -------------------------

import re

def filter_data_driven_candidates(
    de,
    case,
    min_mean_case=0.05,
    min_effect=0.25,
):
    effect_col = f"effect_{case}_vs_Sham"
    mean_col = f"mean_{case}"
    logfc_col = f"log2fc_proxy_{case}_vs_Sham"

    df = de.copy()

    # Remove obvious non-target / housekeeping-style genes
    bad_patterns = [
        r"^Rpl",   # ribosomal large subunit
        r"^Rps",   # ribosomal small subunit
        r"^mt-",   # mitochondrial
        r"^Hba",   # haemoglobin alpha
        r"^Hbb",   # haemoglobin beta
        r"^Gm\d+", # predicted genes
        r"^Mir",   # microRNAs
    ]

    bad_exact = {
        "Malat1",
        "Actb",
        "Gapdh",
        "Tmsb4x",
    }

    pattern = "|".join(bad_patterns)

    df = df[
        ~df["gene"].str.contains(pattern, regex=True, na=False)
        & ~df["gene"].isin(bad_exact)
    ].copy()

    # Keep genes with meaningful expression and stroke induction
    df = df[
        (df[mean_col] >= min_mean_case)
        & (df[effect_col] >= min_effect)
    ].copy()

    # Sort by absolute effect first, then fold-change proxy
    df = df.sort_values(
        [effect_col, logfc_col],
        ascending=False,
    ).reset_index(drop=True)

    return df

In [67]:
# -------------------------
# 7.2 Apply data-driven filtering to D02 and D14
# -------------------------

d02_candidates = filter_data_driven_candidates(
    de=de_d02,
    case="D02",
)

d14_candidates = filter_data_driven_candidates(
    de=de_d14,
    case="D14",
)

print("D02 candidates:", d02_candidates.shape)
print("D14 candidates:", d14_candidates.shape)

print("Top data-driven D02 EC candidates:")
display(d02_candidates.head(50))

print("Top data-driven D14 EC candidates:")
display(d14_candidates.head(50))

D02 candidates: (150, 6)
D14 candidates: (43, 6)
Top data-driven D02 EC candidates:


,gene,mean_D02,mean_Sham,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,pvalue_D02_vs_Sham
0,Igfbp7,6.496007,1.453066,5.042942,2.160453,0.008618
1,Ly6a,6.753810,2.183473,4.570337,1.629077,0.008333
2,Ctla2a,4.136150,0.477916,3.658234,3.113456,0.005249
3,Sparc,4.324855,0.831155,3.493701,2.379462,0.021555
4,Fth1,3.664283,1.307899,2.356383,1.486278,0.002023
5,Ly6c1,5.853538,3.717894,2.135644,0.654823,0.123421
6,Ifitm3,2.267544,0.545412,1.722131,2.055709,0.004427
7,Hsp90ab1,2.911258,1.262513,1.648745,1.205344,0.068693
8,Vim,1.873383,0.365958,1.507425,2.355893,0.012264
9,Calm1,3.666962,2.188762,1.478201,0.744470,0.078412


Top data-driven D14 EC candidates:


,gene,mean_D14,mean_Sham,effect_D14_vs_Sham,log2fc_proxy_D14_vs_Sham,pvalue_D14_vs_Sham
0,Bsg,8.237078,5.933134,2.303945,0.473338,0.319651
1,Itm2a,2.889887,1.651506,1.238381,0.807231,0.235502
2,Pltp,2.359225,1.130960,1.228264,1.060764,0.141996
3,B2m,2.226214,1.035883,1.190331,1.103731,0.090017
4,Cldn5,2.905112,1.813087,1.092024,0.680145,0.279994
5,Hsp90ab1,2.101411,1.262513,0.838899,0.735060,0.176462
6,Jun,1.489664,0.671416,0.818247,1.149706,0.078283
7,Sparcl1,1.935904,1.127995,0.807909,0.779246,0.084158
8,Slco1a4,1.997028,1.372543,0.624485,0.541003,0.217050
9,H2-D1,1.292630,0.674647,0.617983,0.938104,0.058116


In [68]:
# -------------------------
# 7.3 Save data-driven candidate tables
# -------------------------

d02_candidates_path = out_dir / "data_driven_D02_EC_candidates.csv"
d14_candidates_path = out_dir / "data_driven_D14_EC_candidates.csv"

d02_candidates.head(200).to_csv(d02_candidates_path, index=False)
d14_candidates.head(200).to_csv(d14_candidates_path, index=False)

print("Saved D02 data-driven candidates:", d02_candidates_path)
print("Saved D14 data-driven candidates:", d14_candidates_path)

Saved D02 data-driven candidates: ../data/processed/data_driven_D02_EC_candidates.csv
Saved D14 data-driven candidates: ../data/processed/data_driven_D14_EC_candidates.csv


## 8. Annotate data-driven candidates with UniProt surface-accessibility information

The data-driven D02 and D14 candidate lists include many genes that are stroke-responsive in endothelial cells but are not necessarily suitable peptide-shuttle targets. For peptide targeting, the protein should ideally be cell-surface accessible, membrane-associated with an extracellular domain, or otherwise biologically relevant to the extracellular endothelial environment.

This section queries UniProt for the top data-driven candidates and extracts canonical protein annotations, subcellular localisation, and keywords. These annotations are then converted into broad accessibility classes:

- `surface_accessible`: strongest class for direct peptide targeting
- `membrane_unknown_location`: possible target, requires topology/domain inspection
- `secreted_or_extracellular`: useful biology or context marker, but usually not a direct receptor
- `intracellular_or_organelle`: rejected for extracellular peptide targeting
- `unknown`: requires manual inspection

In [69]:
# -------------------------
# 8.1 Define UniProt query and annotation helpers
# -------------------------

import time
import requests

def safe_text(x):
    if x is None or pd.isna(x):
        return ""
    return str(x)


def query_uniprot_mouse_gene(gene):
    url = "https://rest.uniprot.org/uniprotkb/search"
    fields = "accession,gene_names,protein_name,cc_subcellular_location,keyword"

    queries = [
        f"gene_exact:{gene} AND organism_id:10090 AND reviewed:true",
        f"gene_exact:{gene} AND organism_id:10090",
    ]

    for query in queries:
        r = requests.get(
            url,
            params={
                "query": query,
                "format": "json",
                "size": 1,
                "fields": fields,
            },
            timeout=30,
        )
        r.raise_for_status()
        results = r.json().get("results", [])

        if results:
            entry = results[0]
            break
    else:
        return {
            "gene": gene,
            "uniprot_accession": None,
            "protein_name": None,
            "subcellular_location_text": None,
            "keywords_text": None,
            "uniprot_status": "not_found",
        }

    protein_desc = entry.get("proteinDescription", {})
    protein_name = (
        protein_desc
        .get("recommendedName", {})
        .get("fullName", {})
        .get("value")
    )

    if protein_name is None and protein_desc.get("submissionNames"):
        protein_name = (
            protein_desc["submissionNames"][0]
            .get("fullName", {})
            .get("value")
        )

    locations = []

    for comment in entry.get("comments", []):
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for loc in comment.get("subcellularLocations", []):
                pieces = [
                    loc.get("location", {}).get("value"),
                    loc.get("topology", {}).get("value"),
                    loc.get("orientation", {}).get("value"),
                ]
                pieces = [p for p in pieces if p]

                if pieces:
                    locations.append("; ".join(pieces))

    keywords = [
        kw["name"]
        for kw in entry.get("keywords", [])
        if kw.get("name")
    ]

    return {
        "gene": gene,
        "uniprot_accession": entry.get("primaryAccession"),
        "protein_name": protein_name,
        "subcellular_location_text": " | ".join(locations) if locations else None,
        "keywords_text": " | ".join(keywords) if keywords else None,
        "uniprot_status": "found",
    }

In [70]:
# -------------------------
# 8.2 Define surface-accessibility classifier
# -------------------------

def classify_surface_accessibility(location_text, keywords_text):
    text = f"{safe_text(location_text)} {safe_text(keywords_text)}".lower()

    surface_terms = [
        "cell membrane",
        "plasma membrane",
        "cell surface",
        "external side of plasma membrane",
        "extracellular side",
        "gpi-anchor",
        "glycosylphosphatidylinositol",
        "lipid-anchor",
    ]

    extracellular_terms = [
        "secreted",
        "extracellular space",
        "extracellular matrix",
        "extracellular region",
    ]

    intracellular_terms = [
        "mitochondrion",
        "mitochondrial",
        "endoplasmic reticulum",
        "sarcoplasmic reticulum",
        "golgi",
        "lysosome",
        "endosome",
        "cytoplasmic vesicle",
        "nucleus",
        "nuclear",
        "cytoplasm",
        "cytosol",
        "ribosome",
        "peroxisome",
    ]

    has_surface = any(t in text for t in surface_terms)
    has_extra = any(t in text for t in extracellular_terms)
    has_intra = any(t in text for t in intracellular_terms)
    has_membrane = "membrane" in text or "transmembrane" in text

    if has_surface:
        return "surface_accessible"
    if has_extra and has_intra:
        return "secreted_or_extracellular_mixed"
    if has_extra:
        return "secreted_or_extracellular"
    if has_intra:
        return "intracellular_or_organelle"
    if has_membrane:
        return "membrane_unknown_location"

    return "unknown"


def targetability_comment(surface_class):
    comments = {
        "surface_accessible": "best class for direct peptide-shuttle targeting",
        "membrane_unknown_location": "possible target, but needs topology/extracellular-domain check",
        "secreted_or_extracellular": "useful EC/stroke context marker, usually not a direct receptor",
        "secreted_or_extracellular_mixed": "possible extracellular context marker, but not a clean direct receptor",
        "unknown": "manual annotation required",
        "intracellular_or_organelle": "reject for extracellular peptide targeting",
    }

    return comments.get(surface_class, "manual annotation required")


def targetability_score(surface_class):
    scores = {
        "surface_accessible": 4,
        "membrane_unknown_location": 2,
        "secreted_or_extracellular": 1,
        "secreted_or_extracellular_mixed": 0.5,
        "unknown": 0,
        "intracellular_or_organelle": -1,
    }

    return scores.get(surface_class, 0)

In [71]:
# -------------------------
# 8.3 Annotate candidates with UniProt
# -------------------------

def annotate_with_uniprot(df, gene_col="gene", sleep=0.2):
    rows = []

    for gene in df[gene_col].dropna().astype(str).unique():
        try:
            row = query_uniprot_mouse_gene(gene)
        except Exception as e:
            row = {
                "gene": gene,
                "uniprot_accession": None,
                "protein_name": None,
                "subcellular_location_text": None,
                "keywords_text": None,
                "uniprot_status": "error",
                "uniprot_error": str(e),
            }

        rows.append(row)
        time.sleep(sleep)

    ann = pd.DataFrame(rows)

    ann["surface_accessibility"] = ann.apply(
        lambda r: classify_surface_accessibility(
            r.get("subcellular_location_text"),
            r.get("keywords_text"),
        ),
        axis=1,
    )

    ann["targetability_score"] = ann["surface_accessibility"].apply(
        targetability_score
    )

    ann["targetability_comment"] = ann["surface_accessibility"].apply(
        targetability_comment
    )

    return df.merge(ann, on="gene", how="left")


d02_annotated = annotate_with_uniprot(d02_candidates.head(100))
d14_annotated = annotate_with_uniprot(d14_candidates.head(100))

d02_annotated_path = out_dir / "D02_candidates_uniprot_annotated.csv"
d14_annotated_path = out_dir / "D14_candidates_uniprot_annotated.csv"

d02_annotated.to_csv(d02_annotated_path, index=False)
d14_annotated.to_csv(d14_annotated_path, index=False)

print("Saved D02 UniProt annotations:", d02_annotated_path)
print("Saved D14 UniProt annotations:", d14_annotated_path)

Saved D02 UniProt annotations: ../data/processed/D02_candidates_uniprot_annotated.csv
Saved D14 UniProt annotations: ../data/processed/D14_candidates_uniprot_annotated.csv


In [72]:
# -------------------------
# 8.4 Inspect surface-accessibility classes
# -------------------------

print("D02 surface-accessibility classes:")
display(
    d02_annotated["surface_accessibility"]
    .value_counts(dropna=False)
    .to_frame("n_genes")
)

print("D14 surface-accessibility classes:")
display(
    d14_annotated["surface_accessibility"]
    .value_counts(dropna=False)
    .to_frame("n_genes")
)

D02 surface-accessibility classes:


,n_genes
surface_accessibility,
intracellular_or_organelle,41
surface_accessible,31
secreted_or_extracellular,10
unknown,9
secreted_or_extracellular_mixed,6
membrane_unknown_location,3


D14 surface-accessibility classes:


,n_genes
surface_accessibility,
surface_accessible,16
intracellular_or_organelle,12
secreted_or_extracellular,9
membrane_unknown_location,3
secreted_or_extracellular_mixed,3


## 9. Prioritise peptide-shuttle receptor candidates

The D02 response is used as the primary prioritisation window because the aim is to identify early stroke-induced endothelial surface candidates. Candidate genes are first filtered to keep proteins with plausible extracellular or membrane relevance based on UniProt annotation. Manual biological prioritisation is then applied to separate likely peptide-target candidates from secondary candidates, extracellular context markers, and false-positive surface annotations.

The final ranking combines:

- EC pseudobulk expression after D02 stroke,
- D02 induction effect size,
- log2 fold-change proxy,
- UniProt surface-accessibility class,
- and manual biological interpretation.

In [73]:
# -------------------------
# 9.1 Build D02 candidate pool from UniProt-annotated genes
# -------------------------

candidate_classes = [
    "surface_accessible",
    "membrane_unknown_location",
    "secreted_or_extracellular",
    "secreted_or_extracellular_mixed",
    "unknown",
]

candidate_pool = d02_annotated[
    d02_annotated["surface_accessibility"].isin(candidate_classes)
].copy()

print("Candidate pool shape:", candidate_pool.shape)

display(
    candidate_pool[
        [
            "gene",
            "protein_name",
            "mean_D02",
            "effect_D02_vs_Sham",
            "log2fc_proxy_D02_vs_Sham",
            "surface_accessibility",
            "targetability_comment",
        ]
    ].head(20)
)

Candidate pool shape: (59, 14)


,gene,protein_name,mean_D02,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,surface_accessibility,targetability_comment
0,Igfbp7,Insulin-like growth factor-binding protein 7,6.496007,5.042942,2.160453,secreted_or_extracellular,"useful EC/stroke context marker, usually not a..."
1,Ly6a,Lymphocyte antigen 6A-2/6E-1,6.753810,4.570337,1.629077,surface_accessible,best class for direct peptide-shuttle targeting
2,Ctla2a,Protein CTLA-2-alpha,4.136150,3.658234,3.113456,secreted_or_extracellular,"useful EC/stroke context marker, usually not a..."
3,Sparc,SPARC,4.324855,3.493701,2.379462,secreted_or_extracellular,"useful EC/stroke context marker, usually not a..."
5,Ly6c1,Lymphocyte antigen 6C1,5.853538,2.135644,0.654823,surface_accessible,best class for direct peptide-shuttle targeting
6,Ifitm3,Interferon-induced transmembrane protein 3,2.267544,1.722131,2.055709,surface_accessible,best class for direct peptide-shuttle targeting
7,Hsp90ab1,Heat shock protein HSP 90-beta,2.911258,1.648745,1.205344,surface_accessible,best class for direct peptide-shuttle targeting
8,Vim,Vimentin,1.873383,1.507425,2.355893,surface_accessible,best class for direct peptide-shuttle targeting
10,Ybx1,Y-box-binding protein 1,1.875088,1.211122,1.497776,secreted_or_extracellular_mixed,"possible extracellular context marker, but not..."
11,B2m,Beta-2-microglobulin,2.225297,1.189415,1.103137,secreted_or_extracellular,"useful EC/stroke context marker, usually not a..."


In [74]:
# -------------------------
# 9.2 Classify D02 stroke response
# -------------------------

def classify_stroke_response(row):
    effect = row["effect_D02_vs_Sham"]
    logfc = row["log2fc_proxy_D02_vs_Sham"]
    mean_d02 = row["mean_D02"]

    if effect >= 1.0 and logfc >= 1.0:
        return "strong early D02-induced"
    if effect >= 0.5 and logfc >= 0.5:
        return "moderate early D02-induced"
    if mean_d02 >= 1.0 and abs(effect) < 0.5:
        return "highly expressed / preserved"
    if effect > 0:
        return "weakly D02-induced"

    return "not D02-induced"


candidate_pool["stroke_response"] = candidate_pool.apply(
    classify_stroke_response,
    axis=1,
)

display(
    candidate_pool["stroke_response"]
    .value_counts()
    .to_frame("n_genes")
)

,n_genes
stroke_response,
moderate early D02-induced,21
weakly D02-induced,20
strong early D02-induced,13
highly expressed / preserved,5


In [75]:
# -------------------------
# 9.3 Manual biological prioritisation
# -------------------------

primary = {
    "Ly6a", "Ly6e", "Tm4sf1", "Esam",
    "Cd200", "Itgb1", "Fxyd5", "Ecscr",
}

secondary = {
    "Ly6c1", "Ifitm3", "Ifitm2", "Itm2a",
    "Tmem252", "Ramp2", "Cd81", "Pecam1",
}

context = {
    "Igfbp7", "Sparc", "Egfl7", "Edn1", "Col4a1", "Col4a2",
}

reject_surface = {
    "Hsp90ab1", "Vim", "Eef1a1", "Glul", "Hspa5", "Calr",
    "Gnas", "Hspa8", "Rack1", "Msn", "Cdc42", "Cfl1", "Rab11a",
}


def manual_priority(gene):
    if gene in primary:
        return "primary peptide-target candidate"
    if gene in secondary:
        return "secondary candidate / needs validation"
    if gene in context:
        return "stroke EC context marker, not direct receptor"
    if gene in reject_surface:
        return "reject despite surface annotation"

    return "not prioritised"


manual_notes = {
    "Ly6a": "Strongest D02-induced GPI-anchored surface signal; attractive mouse stroke-BBB target but human translation must be checked.",
    "Ly6e": "GPI-anchored surface protein with early induction; possible Ly6-family target.",
    "Tm4sf1": "Transmembrane protein with strong D02 induction; extracellular loops/topology need validation.",
    "Esam": "Endothelial-selective adhesion molecule; surface accessible and EC-relevant.",
    "Cd200": "Single-pass membrane glycoprotein with significant early induction; extracellular domain likely accessible.",
    "Itgb1": "Surface integrin receptor; targetable but broad expression/off-target risk.",
    "Fxyd5": "Single-pass membrane protein with early induction; needs BBB/endothelial specificity check.",
    "Ecscr": "Endothelial cell-specific surface regulator with strong logFC and significant D02 response.",
    "Ly6c1": "Strong surface signal but immune/vascular specificity and translational relevance need caution.",
    "Ifitm3": "Induced membrane protein but more interferon/stress-associated than classic shuttle receptor.",
    "Ifitm2": "Induced membrane protein; similar caution as Ifitm3.",
    "Itm2a": "Membrane protein candidate; extracellular topology and BBB specificity need validation.",
    "Tmem252": "Strong D02 induction; membrane topology and function unclear.",
    "Ramp2": "Surface receptor-modifying protein; relevant but signalling biology may complicate targeting.",
    "Cd81": "Surface tetraspanin; targetable but broad expression risk.",
    "Pecam1": "Endothelial surface marker; useful control/reference but broad vascular expression.",
    "Igfbp7": "Strong EC/stroke marker but secreted, therefore not a clean direct receptor.",
    "Sparc": "Secreted ECM/remodelling marker; useful context marker.",
    "Egfl7": "Secreted endothelial/angiogenic context marker, not direct receptor.",
    "Edn1": "Secreted vasoactive peptide; context marker, not shuttle receptor.",
    "Col4a1": "Extracellular matrix marker, not direct shuttle receptor.",
    "Col4a2": "Extracellular matrix marker, not direct shuttle receptor.",
}


def final_rationale(priority):
    if priority == "primary peptide-target candidate":
        return (
            "prioritised for peptide design; inspect extracellular domain/topology, "
            "internalisation, BBB specificity, and human orthology"
        )

    if priority == "secondary candidate / needs validation":
        return (
            "possible peptide-design target; requires validation of specificity, "
            "topology, or translational relevance"
        )

    if priority == "stroke EC context marker, not direct receptor":
        return (
            "useful for defining stroke endothelial state, but not prioritised "
            "as direct shuttle receptor"
        )

    if priority == "reject despite surface annotation":
        return (
            "not prioritised for peptide design because primary biology is "
            "intracellular/stress/cytoskeletal/signalling"
        )

    return "not prioritised in this toy analysis"


candidate_pool["manual_priority"] = candidate_pool["gene"].apply(
    manual_priority
)

candidate_pool["manual_interpretation"] = (
    candidate_pool["gene"]
    .map(manual_notes)
    .fillna("")
)

candidate_pool["final_peptide_design_rationale"] = (
    candidate_pool["manual_priority"]
    .apply(final_rationale)
)

In [76]:
# -------------------------
# 9.4 Compute omics priority score and final manual ranking
# -------------------------

priority_order = {
    "primary peptide-target candidate": 1,
    "secondary candidate / needs validation": 2,
    "stroke EC context marker, not direct receptor": 3,
    "reject despite surface annotation": 4,
    "not prioritised": 5,
}

candidate_pool["priority_order"] = candidate_pool["manual_priority"].map(
    priority_order
)

candidate_pool["omics_priority_score"] = (
    candidate_pool["mean_D02"].rank(pct=True)
    + candidate_pool["effect_D02_vs_Sham"].rank(pct=True)
    + candidate_pool["log2fc_proxy_D02_vs_Sham"].rank(pct=True)
)

final_manual_table = candidate_pool.sort_values(
    ["priority_order", "omics_priority_score"],
    ascending=[True, False],
).copy()

final_columns = [
    "gene",
    "protein_name",
    "mean_D02",
    "mean_Sham",
    "effect_D02_vs_Sham",
    "log2fc_proxy_D02_vs_Sham",
    "pvalue_D02_vs_Sham",
    "omics_priority_score",
    "stroke_response",
    "subcellular_location_text",
    "surface_accessibility",
    "targetability_comment",
    "manual_priority",
    "manual_interpretation",
    "final_peptide_design_rationale",
    "uniprot_accession",
]

display(final_manual_table[final_columns].head(50))

,gene,protein_name,mean_D02,mean_Sham,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,pvalue_D02_vs_Sham,omics_priority_score,stroke_response,subcellular_location_text,surface_accessibility,targetability_comment,manual_priority,manual_interpretation,final_peptide_design_rationale,uniprot_accession
1,Ly6a,Lymphocyte antigen 6A-2/6E-1,6.753810,2.183473,4.570337,1.629077,0.008333,2.593220,strong early D02-induced,"Cell membrane; Lipid-anchor, GPI-anchor",surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Strongest D02-induced GPI-anchored surface sig...,prioritised for peptide design; inspect extrac...,P05533
14,Tm4sf1,Transmembrane 4 L6 family member 1,1.421943,0.355311,1.066632,2.000707,0.003731,2.288136,strong early D02-induced,Membrane; Multi-pass membrane protein,membrane_unknown_location,"possible target, but needs topology/extracellu...",primary peptide-target candidate,Transmembrane protein with strong D02 inductio...,prioritised for peptide design; inspect extrac...,Q64302
25,Cd200,OX-2 membrane glycoprotein,1.035134,0.317887,0.717247,1.703227,0.000703,2.000000,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Single-pass membrane glycoprotein with signifi...,prioritised for peptide design; inspect extrac...,O54901
16,Ly6e,Lymphocyte antigen 6E,2.406169,1.352209,1.053960,0.831419,0.108692,1.779661,moderate early D02-induced,"Cell membrane; Lipid-anchor, GPI-anchor",surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,GPI-anchored surface protein with early induct...,prioritised for peptide design; inspect extrac...,Q64253
37,Ecscr,Endothelial cell-specific chemotaxis regulator,0.639574,0.083505,0.556069,2.937155,0.000125,1.677966,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Endothelial cell-specific surface regulator wi...,prioritised for peptide design; inspect extrac...,Q3TZW0
34,Itgb1,Integrin beta-1,0.876678,0.295314,0.581364,1.569795,0.016298,1.559322,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Surface integrin receptor; targetable but broa...,prioritised for peptide design; inspect extrac...,P09055
36,Esam,Endothelial cell-selective adhesion molecule,0.978323,0.419158,0.559165,1.222813,0.029184,1.440678,moderate early D02-induced,"Cell junction, adherens junction | Cell juncti...",surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Endothelial-selective adhesion molecule; surfa...,prioritised for peptide design; inspect extrac...,Q925F2
54,Fxyd5,FXYD domain-containing ion transport regulator 5,0.920225,0.445696,0.474529,1.045925,0.030093,1.118644,weakly D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,best class for direct peptide-shuttle targeting,primary peptide-target candidate,Single-pass membrane protein with early induct...,prioritised for peptide design; inspect extrac...,P97808
6,Ifitm3,Interferon-induced transmembrane protein 3,2.267544,0.545412,1.722131,2.055709,0.004427,2.593220,strong early D02-induced,Cell membrane; Single-pass type II membrane pr...,surface_accessible,best class for direct peptide-shuttle targeting,secondary candidate / needs validation,Induced membrane protein but more interferon/s...,possible peptide-design target; requires valid...,Q9CQW9
12,Tmem252,Transmembrane protein 252,1.471563,0.291582,1.179982,2.335374,0.000901,2.457627,strong early D02-induced,Membrane; Multi-pass membrane protein,membrane_unknown_location,"possible target, but needs topology/extracellu...",secondary candidate / needs validation,Strong D02 induction; me

In [77]:
# -------------------------
# 9.5 Save prioritised candidate table
# -------------------------

final_manual_path = out_dir / "FINAL_manual_prioritised_EC_peptide_target_candidates.csv"

final_manual_table[final_columns].to_csv(
    final_manual_path,
    index=False,
)

print("Saved prioritised EC peptide-target candidates:", final_manual_path)

Saved prioritised EC peptide-target candidates: ../data/processed/FINAL_manual_prioritised_EC_peptide_target_candidates.csv


## 10. Export final receptor-candidate report table

The final report table reformats the prioritised candidates into a compact, human-readable output for downstream peptide-design notebooks. This table includes the gene candidate, UniProt protein annotation, EC pseudobulk effect statistics, surface-accessibility class, biological interpretation, peptide-design rationale, and caveats.

The output remains a gene-level receptor-candidate shortlist. It does not yet validate isoform-specific topology, extracellular-domain structure, internalisation, human translation, or binding suitability.

In [78]:
# -------------------------
# 10.1 Build final report table
# -------------------------

final_report_table = final_manual_table.copy()

final_report_table = final_report_table.rename(columns={
    "gene": "gene_candidate",
    "protein_name": "uniprot_canonical_protein_annotation",
    "subcellular_location_text": "uniprot_subcellular_location",
    "surface_accessibility": "putative_accessibility_class",
    "manual_priority": "target_prioritisation_class",
    "manual_interpretation": "biological_interpretation",
})

final_report_table["evidence_level"] = (
    "gene-level transcriptomic candidate with UniProt canonical protein annotation; "
    "requires isoform, extracellular-domain, receptor-complex, mouse-human orthology, "
    "and internalisation validation"
)

final_report_columns = [
    "gene_candidate",
    "uniprot_canonical_protein_annotation",
    "mean_D02",
    "mean_Sham",
    "effect_D02_vs_Sham",
    "log2fc_proxy_D02_vs_Sham",
    "pvalue_D02_vs_Sham",
    "omics_priority_score",
    "stroke_response",
    "uniprot_subcellular_location",
    "putative_accessibility_class",
    "target_prioritisation_class",
    "biological_interpretation",
    "final_peptide_design_rationale",
    "evidence_level",
    "uniprot_accession",
]

display(final_report_table[final_report_columns].head(30))

,gene_candidate,uniprot_canonical_protein_annotation,mean_D02,mean_Sham,effect_D02_vs_Sham,log2fc_proxy_D02_vs_Sham,pvalue_D02_vs_Sham,omics_priority_score,stroke_response,uniprot_subcellular_location,putative_accessibility_class,target_prioritisation_class,biological_interpretation,final_peptide_design_rationale,evidence_level,uniprot_accession
1,Ly6a,Lymphocyte antigen 6A-2/6E-1,6.753810,2.183473,4.570337,1.629077,0.008333,2.593220,strong early D02-induced,"Cell membrane; Lipid-anchor, GPI-anchor",surface_accessible,primary peptide-target candidate,Strongest D02-induced GPI-anchored surface sig...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,P05533
14,Tm4sf1,Transmembrane 4 L6 family member 1,1.421943,0.355311,1.066632,2.000707,0.003731,2.288136,strong early D02-induced,Membrane; Multi-pass membrane protein,membrane_unknown_location,primary peptide-target candidate,Transmembrane protein with strong D02 inductio...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,Q64302
25,Cd200,OX-2 membrane glycoprotein,1.035134,0.317887,0.717247,1.703227,0.000703,2.000000,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,primary peptide-target candidate,Single-pass membrane glycoprotein with signifi...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,O54901
16,Ly6e,Lymphocyte antigen 6E,2.406169,1.352209,1.053960,0.831419,0.108692,1.779661,moderate early D02-induced,"Cell membrane; Lipid-anchor, GPI-anchor",surface_accessible,primary peptide-target candidate,GPI-anchored surface protein with early induct...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,Q64253
37,Ecscr,Endothelial cell-specific chemotaxis regulator,0.639574,0.083505,0.556069,2.937155,0.000125,1.677966,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,primary peptide-target candidate,Endothelial cell-specific surface regulator wi...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,Q3TZW0
34,Itgb1,Integrin beta-1,0.876678,0.295314,0.581364,1.569795,0.016298,1.559322,moderate early D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,primary peptide-target candidate,Surface integrin receptor; targetable but broa...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,P09055
36,Esam,Endothelial cell-selective adhesion molecule,0.978323,0.419158,0.559165,1.222813,0.029184,1.440678,moderate early D02-induced,"Cell junction, adherens junction | Cell juncti...",surface_accessible,primary peptide-target candidate,Endothelial-selective adhesion molecule; surfa...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,Q925F2
54,Fxyd5,FXYD domain-containing ion transport regulator 5,0.920225,0.445696,0.474529,1.045925,0.030093,1.118644,weakly D02-induced,Cell membrane; Single-pass type I membrane pro...,surface_accessible,primary peptide-target candidate,Single-pass membrane protein with early induct...,prioritised for peptide design; inspect extrac...,gene-level transcriptomic candidate with UniPr...,P97808
6,Ifitm3,Interferon-induced transmembrane protein 3,2.267544,0.545412,1.722131,2.055709,0.004427,2.593220,strong early D02-induced,Cell membrane; Single-pass type II membrane pr...,surface_accessible,secondary candidate / needs validation,Induced membrane protein but more interferon/s...,possible peptide-design target; requires valid...,gene-level transcriptomic candidate with UniPr...,Q9CQW9
12,Tmem252,Transmembrane protein 252,1.471563,0.291582,1.179982,2.335374,0.000901,2.457627,strong early D02-induced,Membrane; Multi-pass membrane protein,membrane_unknown_location,secondary candidate / needs validation,Strong D0

In [79]:
# -------------------------
# 10.2 Save final report table
# -------------------------

final_report_path = out_dir / "FINAL_gene_level_EC_peptide_target_candidates_with_caveats.csv"

final_report_table[final_report_columns].to_csv(
    final_report_path,
    index=False,
)

print("Saved final gene-level EC peptide-target candidate report:", final_report_path)

Saved final gene-level EC peptide-target candidate report: ../data/processed/FINAL_gene_level_EC_peptide_target_candidates_with_caveats.csv


Primary candidates identified in this run included `Ly6a`, `Tm4sf1`, `Ly6e`, `Ecscr`, `Cd200`, `Itgb1`, `Esam`, and `Fxyd5`. Secondary and context candidates were retained for interpretation and future expansion.

## Final note

This notebook exports the receptor-candidate shortlist used by the downstream peptide-design notebooks. The prioritised candidates are gene-level hypotheses, not validated shuttle receptors.

The final report table is saved as:

`../data/processed/FINAL_gene_level_EC_peptide_target_candidates_with_caveats.csv`